# Extrinsic Downstream Task Evaluation
## Sinhala Script Robustness — Unicode vs Romanized

## 1. Configuration

In [ ]:
# ============================================================
# MODEL CONFIGURATION — CHANGE ONLY THIS CELL TO SWITCH MODELS
# ============================================================

MODEL_ID    = "microsoft/phi-4"    # HuggingFace model ID
MODEL_LABEL = "Phi-4"              # Short label for filenames & tables
MAX_NEW_TOKENS = 40                 # Max tokens to generate per response
TEMPERATURE    = 0.0                # Greedy decoding (deterministic, zero-shot)
MAX_INPUT_TOKENS = 4096             # Truncate prompts longer than this (safety margin)

# Set True only for large models that won't fit in fp16 across 2x T4 (32GB total).
LOAD_IN_8BIT = False

# If a previous run's CSV for this model+dataset already exists, skip ids already completed
# instead of starting over. Protects against Kaggle session interruptions on a ~4,451-item run.
RESUME = True

# Dataset paths (update if your Kaggle dataset mount differs)
DATA_DIR = "/kaggle/input/sinhala-eval-datasets"  # Folder containing the 3 JSONL files

# Output prefix (auto-generated from MODEL_LABEL)
import re
OUTPUT_PREFIX = re.sub(r"[^a-z0-9]+", "_", MODEL_LABEL.lower()).strip("_")

print(f"Model:        {MODEL_ID}")
print(f"Label:        {MODEL_LABEL}")
print(f"8-bit:        {LOAD_IN_8BIT}")
print(f"Resume:       {RESUME}")
print(f"Output:       {OUTPUT_PREFIX}_*.csv")

## 2. Environment Setup & Hardware Verification

In [ ]:
import sys
import os

!{sys.executable} -m pip install --upgrade --quiet transformers accelerate huggingface_hub bitsandbytes

import torch
import json
import csv
import time
import re
import transformers
from pathlib import Path
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Hardware check
device_count = torch.cuda.device_count()
print(f"PyTorch {torch.__version__} | transformers {transformers.__version__} | {device_count} GPU(s)")
for i in range(device_count):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f"  cuda:{i} -> {name} ({mem:.1f} GB)")
assert device_count > 0, "No GPU detected. Enable GPU T4 x2 in Session options."

In [ ]:
# HuggingFace authentication (needed for gated models, e.g. Llama family if ever cloze-scored)
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Authenticated via Kaggle Secrets.")
except Exception as e:
    HF_TOKEN = ""   # Paste your token here if Kaggle Secrets is unavailable
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("Authenticated via explicit token.")
    else:
        print(f"No HF_TOKEN found ({e}). Public models will still work.")

## 3. Load Model & Tokenizer

In [ ]:
print(f"Loading {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN if HF_TOKEN else None,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

has_template = hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None
print(f"Chat template available: {has_template}")
if not has_template:
    print("WARNING: this model has no chat template. It will be prompted with the raw text "
          "instead of its native assistant format — confirm this model is actually instruct-tuned "
          "before trusting the results (this notebook is not meant for base models).")

model_kwargs = dict(
    device_map="auto",
    token=HF_TOKEN if HF_TOKEN else None,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
if LOAD_IN_8BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
else:
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
model.eval()
model.config.use_cache = True  # Enable KV cache for generation

print("Model loaded successfully.")
print(f"Precision: {model.dtype}")
if hasattr(model, "hf_device_map"):
    print(f"Device map: {model.hf_device_map}")

## 4. Load Evaluation Datasets

In [ ]:
def load_jsonl(filepath):
    """Load a JSONL file into a list of dicts."""
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

mmlu_data = load_jsonl(os.path.join(DATA_DIR, "sinhala_mmlu.jsonl"))
sold_data = load_jsonl(os.path.join(DATA_DIR, "sold.jsonl"))
piqa_data = load_jsonl(os.path.join(DATA_DIR, "global_piqa.jsonl"))

print(f"Sinhala MMLU : {len(mmlu_data):,} items (4-way MCQ)")
print(f"SOLD         : {len(sold_data):,} items (binary classification)")
print(f"Global PIQA  : {len(piqa_data):,} items (2-way MCQ)")
print(f"\nTotal items  : {len(mmlu_data) + len(sold_data) + len(piqa_data):,} "
      f"x 2 script conditions = {(len(mmlu_data) + len(sold_data) + len(piqa_data)) * 2:,} prompts")

## 5. Prompt Templates & Label Extraction

In [ ]:
# ============================================================
# PROMPT TEMPLATES
# ============================================================

def _humanize_domain(domain):
    """'Social_Science' -> 'Social Science'. Leaves single-word domains untouched."""
    return str(domain).replace("_", " ")


def build_mcq_prompt(question, options, subject=None):
    """
    Build the zero-shot numeric-answer MCQ prompt (Sinhala MMLU or Global PIQA).
    Options are numbered 1..N. Returns (prompt, digit_labels) where digit_labels
    is ["1", "2", ...] in the same order as `options`.
    """
    digit_labels = [str(i + 1) for i in range(len(options))]
    options_str = "\n".join(f"{d}. {opt}" for d, opt in zip(digit_labels, options))

    if subject is not None:
        # Sinhala MMLU wording
        prompt = (
            f"This is a multiple-choice question related to the {_humanize_domain(subject)}. "
            f"Choose the correct or most appropriate answer from answers 1, 2, 3, or 4 for the "
            f"following question.\n"
            f"Question: {question}\n"
            f"{options_str}\n"
            f"Answer:"
        )
    else:
        # Global PIQA wording. The spec's wording has no answer-format instruction, so
        # "Answer with only 1 or 2." is appended — needed for the extractor, not in the
        # original spec text.
        prompt = (
            f"Out of the given two answers, which one is the best option to complete the given text?\n"
            f"{question}\n"
            f"{options_str}\n"
            f"Answer with only 1 or 2.\n"
            f"Answer:"
        )
    return prompt, digit_labels


def build_sold_prompt(text):
    """Zero-shot binary classification prompt for offensive language detection."""
    return (
        f"Classify the following Sinhala text as either 'NOT' (not offensive) or 'OFF' (offensive).\n\n"
        f"Text: {text}\n\n"
        f"Respond with only 'NOT' or 'OFF'.\n"
        f"Answer:"
    )


# ============================================================
# LABEL EXTRACTION
# ============================================================

def _extract_token(generated_text, valid_tokens, keyword_pattern):
    """
    Shared boundary-safe extractor for both digit tokens ("1".."4") and word tokens
    ("NOT"/"OFF"). `keyword_pattern` is the regex alternation for words that typically
    precede the answer (e.g. "answer|option|choice" or "label|answer|classification").
    """
    if not generated_text:
        return "INVALID"
    text = generated_text.strip()
    pattern_tokens = "|".join(re.escape(t) for t in valid_tokens)
    # Digits must not be adjacent to other digits (so "10" can't match token "1").
    is_digit = valid_tokens and valid_tokens[0].isdigit()
    boundary = r"(?<!\d)" if is_digit else r"\b"
    end_boundary = r"(?!\d)" if is_digit else r"\b"

    # 1. Leading answer: "1", "1.", "(1)", "**1**", "NOT", "Answer: NOT"
    prefix_match = re.match(
        rf"^(?:(?:the\s+)?(?:{keyword_pattern})\s*(?:is|=|:)?\s*)?"
        rf"[\*\(#\[\s]*{boundary}({pattern_tokens}){end_boundary}",
        text, re.IGNORECASE
    )
    if prefix_match:
        return prefix_match.group(1).upper()

    # 2. Explicit declaration in the body: "the answer is 2", "answer is NOT"
    body_match = re.search(
        rf"(?:(?:the\s+)?(?:{keyword_pattern})\s*(?:is|=|:)?\s*)"
        rf"[\*\(#\[\s]*{boundary}({pattern_tokens}){end_boundary}",
        text, re.IGNORECASE
    )
    if body_match:
        return body_match.group(1).upper()

    # 3. Standalone token anywhere, with a boundary on both sides.
    standalone_match = re.search(rf"{boundary}({pattern_tokens}){end_boundary}", text, re.IGNORECASE)
    if standalone_match:
        return standalone_match.group(1).upper()

    return "INVALID"


def extract_mcq_choice(generated_text, digit_labels):
    """Extract the chosen digit ("1".."4") from generated text."""
    return _extract_token(generated_text, digit_labels, r"correct\s+answer|answer|option|choice")


def digit_to_letter(digit):
    """"1" -> "A", "2" -> "B", ... "INVALID" passes through unchanged."""
    if digit.isdigit() and 1 <= int(digit) <= 26:
        return chr(64 + int(digit))
    return digit


def extract_sold_label(generated_text):
    """Extract NOT or OFF with whole-word boundary checks."""
    return _extract_token(generated_text, ["NOT", "OFF"], "label|answer|classification")


print("Prompt templates and extraction functions defined.")

# Sanity check
p_mcq, digits = build_mcq_prompt("Sample question?", ["opt1", "opt2", "opt3", "opt4"], subject="Social_Science")
print(f"\n--- Sample MMLU Prompt ---\n{p_mcq}")
p_piqa, _ = build_mcq_prompt("Sample question?", ["opt1", "opt2"])
print(f"\n--- Sample PIQA Prompt ---\n{p_piqa}")
print(f"\n--- Sample SOLD Prompt ---\n{build_sold_prompt('Sample text')}")
print(f"\ndigit_to_letter test: 1->{digit_to_letter('1')} 2->{digit_to_letter('2')} INVALID->{digit_to_letter('INVALID')}")

## 6. Generation Engine with Native Chat Template

In [ ]:
@torch.inference_mode()
def generate_response(prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
    """Generate a response from the instruct model given a plain-text prompt."""
    if hasattr(tokenizer, "apply_chat_template") and getattr(tokenizer, "chat_template", None):
        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        formatted_prompt = prompt

    device = next(model.parameters()).device
    inputs = tokenizer(
        formatted_prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS
    ).to(device)
    input_len = inputs["input_ids"].shape[1]

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": temperature > 0,
        "pad_token_id": tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
    }
    if temperature > 0:
        gen_kwargs["temperature"] = temperature

    outputs = model.generate(**inputs, **gen_kwargs)
    generated_ids = outputs[0][input_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return generated_text, input_len, len(generated_ids)

print("Generation engine initialized.")

## 7. Pilot Run

In [ ]:
print("=" * 75)
print("PILOT RUN — 5 items per task per script condition")
print("=" * 75)

print("\n--- Sinhala MMLU (MCQ, numeric answer) ---")
for item in mmlu_data[:5]:
    for script in ["unicode", "romanized"]:
        s = item[script]
        prompt, digits = build_mcq_prompt(s["text"], s["options"], subject=item.get("strata", {}).get("domain"))
        gen_text, n_in, n_out = generate_response(prompt)
        pred_digit = extract_mcq_choice(gen_text, digits)
        pred_label = digit_to_letter(pred_digit)
        correct = "\u2713" if pred_label == item["label"] else "\u2717"
        print(f"  [{script:9}] id={item['id']} | gold={item['label']} pred={pred_digit}->{pred_label} {correct} | raw={gen_text[:60]!r}")

print("\n--- SOLD (Binary) ---")
for item in sold_data[:5]:
    for script in ["unicode", "romanized"]:
        s = item[script]
        prompt = build_sold_prompt(s["text"])
        gen_text, n_in, n_out = generate_response(prompt)
        pred = extract_sold_label(gen_text)
        correct = "\u2713" if pred == item["label"] else "\u2717"
        print(f"  [{script:9}] id={item['id']} | gold={item['label']} pred={pred} {correct} | raw={gen_text[:60]!r}")

print("\n--- Global PIQA (MCQ, numeric answer) ---")
for item in piqa_data[:5]:
    for script in ["unicode", "romanized"]:
        s = item[script]
        prompt, digits = build_mcq_prompt(s["text"], s["options"])
        gen_text, n_in, n_out = generate_response(prompt)
        pred_digit = extract_mcq_choice(gen_text, digits)
        pred_label = digit_to_letter(pred_digit)
        correct = "\u2713" if pred_label == item["label"] else "\u2717"
        print(f"  [{script:9}] id={item['id']} | gold={item['label']} pred={pred_digit}->{pred_label} {correct} | raw={gen_text[:60]!r}")

print("\n" + "=" * 75)
print("Pilot complete. Check output above for clean extractions before running Section 8.")
print("=" * 75)

## 8. Full Evaluation

In [ ]:
MCQ_FIELDNAMES_BASE = [
    "id", "gold_label",
    "unicode_pred_digit", "unicode_pred_label", "unicode_correct", "unicode_raw_output",
    "unicode_n_input_tokens", "unicode_n_output_tokens",
    "romanized_pred_digit", "romanized_pred_label", "romanized_correct", "romanized_raw_output",
    "romanized_n_input_tokens", "romanized_n_output_tokens",
]
SOLD_FIELDNAMES_BASE = [
    "id", "gold_label",
    "unicode_pred", "unicode_correct", "unicode_raw_output",
    "unicode_n_input_tokens", "unicode_n_output_tokens",
    "romanized_pred", "romanized_correct", "romanized_raw_output",
    "romanized_n_input_tokens", "romanized_n_output_tokens",
]


def _existing_ids(output_csv):
    """Ids already written by a previous (possibly interrupted) run."""
    if not (RESUME and os.path.exists(output_csv)):
        return set()
    with open(output_csv, "r", newline="", encoding="utf-8-sig") as f:
        return {row["id"] for row in csv.DictReader(f)}


def run_mcq_evaluation(dataset, dataset_name, output_csv, strata_fields, subject_field=None):
    """
    Run MCQ evaluation (MMLU or PIQA) on both script conditions.
    strata_fields: keys inside each item's "strata" dict to copy into the CSV as columns
        (e.g. "domain", "difficulty", "culturally_specific"). data/eval/*.jsonl nests these
        under item["strata"], not at the item's top level.
    subject_field: key inside item["strata"] holding the MMLU subject/domain, or None for
        PIQA (no subject wording in its prompt).
    """
    fieldnames = strata_fields + MCQ_FIELDNAMES_BASE
    done_ids = _existing_ids(output_csv)
    mode = "a" if done_ids else "w"

    print(f"\n{'='*65}")
    print(f"Evaluating: {dataset_name} ({len(dataset)} items, {len(done_ids)} already done)")
    print(f"Output:     {output_csv}")
    print(f"{'='*65}")

    with open(output_csv, mode=mode, newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if mode == "w":
            writer.writeheader()

        t0 = time.time()
        remaining = [it for it in dataset if it["id"] not in done_ids]
        for i, item in enumerate(tqdm(remaining, desc=dataset_name)):
            strata = item.get("strata", {})
            row = {k: strata.get(k) for k in strata_fields}
            row["id"] = item["id"]
            row["gold_label"] = item["label"]

            for script in ["unicode", "romanized"]:
                s = item[script]
                subject = strata.get(subject_field) if subject_field else None
                prompt, digits = build_mcq_prompt(s["text"], s["options"], subject=subject)
                gen_text, n_in, n_out = generate_response(prompt)
                pred_digit = extract_mcq_choice(gen_text, digits)
                pred_label = digit_to_letter(pred_digit)
                correct = int(pred_label == item["label"])

                row[f"{script}_pred_digit"] = pred_digit
                row[f"{script}_pred_label"] = pred_label
                row[f"{script}_correct"] = correct
                row[f"{script}_raw_output"] = gen_text.strip()[:200]
                row[f"{script}_n_input_tokens"] = n_in
                row[f"{script}_n_output_tokens"] = n_out

            writer.writerow(row)
            if (i + 1) % 50 == 0:
                f.flush()

        elapsed = time.time() - t0

    n_done = len(remaining)
    if n_done:
        print(f"Done {n_done:,} new items in {elapsed/60:.1f} min ({elapsed/n_done:.2f} s/item)")
    else:
        print("Nothing to do, all items already completed.")
    return output_csv


def run_sold_evaluation(dataset, output_csv):
    """Run SOLD binary classification on both script conditions."""
    fieldnames = ["label"] + SOLD_FIELDNAMES_BASE  # gold label doubles as the strata field
    done_ids = _existing_ids(output_csv)
    mode = "a" if done_ids else "w"

    print(f"\n{'='*65}")
    print(f"Evaluating: SOLD ({len(dataset)} items, {len(done_ids)} already done)")
    print(f"Output:     {output_csv}")
    print(f"{'='*65}")

    with open(output_csv, mode=mode, newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if mode == "w":
            writer.writeheader()

        t0 = time.time()
        remaining = [it for it in dataset if it["id"] not in done_ids]
        for i, item in enumerate(tqdm(remaining, desc="SOLD")):
            row = {"id": item["id"], "gold_label": item["label"], "label": item["label"]}

            for script in ["unicode", "romanized"]:
                s = item[script]
                prompt = build_sold_prompt(s["text"])
                gen_text, n_in, n_out = generate_response(prompt)
                pred = extract_sold_label(gen_text)
                correct = int(pred == item["label"])

                row[f"{script}_pred"] = pred
                row[f"{script}_correct"] = correct
                row[f"{script}_raw_output"] = gen_text.strip()[:200]
                row[f"{script}_n_input_tokens"] = n_in
                row[f"{script}_n_output_tokens"] = n_out

            writer.writerow(row)
            if (i + 1) % 50 == 0:
                f.flush()

        elapsed = time.time() - t0

    n_done = len(remaining)
    if n_done:
        print(f"Done {n_done:,} new items in {elapsed/60:.1f} min ({elapsed/n_done:.2f} s/item)")
    else:
        print("Nothing to do, all items already completed.")
    return output_csv

print("Evaluation runners ready.")

In [ ]:
# ============================================================
# RUN ALL THREE EVALUATIONS
# ============================================================

mmlu_csv = run_mcq_evaluation(
    mmlu_data, "Sinhala MMLU", f"{OUTPUT_PREFIX}_sinhala_mmlu.csv",
    strata_fields=["domain", "difficulty"], subject_field="domain"
)

sold_csv = run_sold_evaluation(
    sold_data, f"{OUTPUT_PREFIX}_sold.csv"
)

piqa_csv = run_mcq_evaluation(
    piqa_data, "Global PIQA", f"{OUTPUT_PREFIX}_global_piqa.csv",
    strata_fields=["culturally_specific"], subject_field=None
)

print("\n" + "="*65)
print("ALL EVALUATIONS COMPLETE")
print("="*65)

## 9. Overall Summary

In [ ]:
import pandas as pd

def summarize(csv_path, task_name):
    df = pd.read_csv(csv_path)
    n = len(df)
    u_acc = df["unicode_correct"].mean() * 100
    r_acc = df["romanized_correct"].mean() * 100
    u_invalid_col = "unicode_pred_label" if "unicode_pred_label" in df.columns else "unicode_pred"
    r_invalid_col = "romanized_pred_label" if "romanized_pred_label" in df.columns else "romanized_pred"
    u_invalid = (df[u_invalid_col] == "INVALID").mean() * 100
    r_invalid = (df[r_invalid_col] == "INVALID").mean() * 100
    gap = u_acc - r_acc

    print(f"{task_name:14} n={n:5,}  Unicode={u_acc:6.2f}%  Romanized={r_acc:6.2f}%  "
          f"Gap={gap:+6.2f}pp  Invalid(U/R)={u_invalid:.1f}%/{r_invalid:.1f}%")
    return {"Task": task_name, "N": n, "Unicode Acc (%)": round(u_acc, 2),
            "Romanized Acc (%)": round(r_acc, 2), "Gap (pp)": round(gap, 2),
            "Unicode Invalid (%)": round(u_invalid, 2), "Romanized Invalid (%)": round(r_invalid, 2)}


print(f"{MODEL_LABEL} \u2014 Extrinsic Evaluation Summary")
print("-" * 90)
results = [
    summarize(mmlu_csv, "Sinhala MMLU"),
    summarize(sold_csv, "SOLD"),
    summarize(piqa_csv, "Global PIQA"),
]
summary_df = pd.DataFrame(results)
summary_csv = f"{OUTPUT_PREFIX}_extrinsic_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"\nSummary saved to: {summary_csv}")
summary_df

## 10. Output File Listing

In [ ]:
print("All output files:")
for f in [mmlu_csv, sold_csv, piqa_csv, summary_csv]:
    size = os.path.getsize(f) / 1024
    print(f"  {f} ({size:.1f} KB)")

print(f"\n\u2705 {MODEL_LABEL} extrinsic evaluation complete!")
print("\nTo evaluate another model:")
print("  1. Change MODEL_ID / MODEL_LABEL (and LOAD_IN_8BIT if it's a large model) in the Configuration cell")
print("  2. Restart the runtime and run all cells")